# RAG end-to-end

Para que quepa en una clase, usamos 3 papers de arxiv en lugar de 50 PDFs. **El pipeline es exactamente el mismo a escala**.

## Pipeline

```
PDFs → Loader → Chunker → Embedder → Chroma (con metadata)
                                            ↓
Query → Retriever (top-20) → Reranker (top-3) → LLM → Respuesta + fuentes
```

## Stack

- Loader: `PyPDFLoader`
- Chunker: `RecursiveCharacterTextSplitter`
- Embedder: `text-embedding-3-small` (OpenAI)
- Vectorstore: Chroma (persistente)
- Reranker: Cohere `rerank-v3.5`
- LLM: `gpt-5-mini`
- Orquestación: Langchain LCEL

## Setup

In [1]:
%pip install -q langchain langchain-core langchain-community langchain-openai langchain-chroma langchain-cohere openai cohere pypdf chromadb tiktoken requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 

In [2]:
import os, getpass, requests
from pathlib import Path

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key (para reranker): ")

OpenAI API Key: ··········
Cohere API Key (para reranker): ··········


## 1. Corpus · 3 papers de arxiv

Descargamos 3 PDFs reales sobre RAG y atención. Cabe en RAM, en disco, y en una clase de 4h. La mecánica del pipeline no cambia con 3 o con 3000.

In [3]:
CORPUS_DIR = Path("./rag_corpus")
CORPUS_DIR.mkdir(exist_ok=True)

PAPERS = {
    "attention.pdf": "https://arxiv.org/pdf/1706.03762.pdf",   # Attention is All You Need
    "rag.pdf":       "https://arxiv.org/pdf/2005.11401.pdf",   # Original RAG paper
    "selfrag.pdf":   "https://arxiv.org/pdf/2310.11511.pdf",   # Self-RAG
}

for name, url in PAPERS.items():
    dest = CORPUS_DIR / name
    if not dest.exists():
        print(f"  ↓ {name}")
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
        r.raise_for_status()
        dest.write_bytes(r.content)
    print(f"  ✓ {name}  ({dest.stat().st_size // 1024} KB)")

  ↓ attention.pdf
  ✓ attention.pdf  (2163 KB)
  ↓ rag.pdf
  ✓ rag.pdf  (864 KB)
  ↓ selfrag.pdf
  ✓ selfrag.pdf  (1372 KB)


## 2. Loading · `PyPDFLoader`

Cargamos cada PDF y guardamos el origen + página como **metadata**. Esa metadata es la que luego permite citar fuentes y filtrar en queries.

In [4]:
from langchain_community.document_loaders import PyPDFLoader

all_docs = []
for name in PAPERS:
    pages = PyPDFLoader(str(CORPUS_DIR / name)).load()
    for page in pages:
        # Enriquecemos la metadata · estos campos los usaremos al filtrar y citar
        page.metadata["source"] = name
        # PyPDFLoader ya rellena 'page' (0-indexed)
    all_docs.extend(pages)
    print(f"  ✓ {name}  · {len(pages)} páginas")

print(f"\nTotal: {len(all_docs)} páginas en {len(PAPERS)} PDFs")

  ✓ attention.pdf  · 15 páginas
  ✓ rag.pdf  · 19 páginas
  ✓ selfrag.pdf  · 30 páginas

Total: 64 páginas en 3 PDFs


## 3. Chunking · `RecursiveCharacterTextSplitter`

Cortamos cada página en chunks de ~1000 caracteres con overlap de 200. Recursive corta intentando respetar saltos de párrafo y oraciones (mejor que fixed-size).

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(all_docs)

print(f"✓ {len(chunks)} chunks creados (de {len(all_docs)} páginas)")
print("\nEjemplo de chunk:")
print(f"metadata: {chunks[0].metadata}")
print(f"texto[:200]: {chunks[0].page_content[:200]}...")

✓ 283 chunks creados (de 64 páginas)

Ejemplo de chunk:
metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}
texto[:200]: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...


## 4. Indexación · Chroma persistente

Embedemos con `text-embedding-3-small` y guardamos en Chroma con persistencia local. La primera vez tarda (genera embeddings); las siguientes carga del disco.

In [7]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

PERSIST_DIR = "./rag_chroma_db"
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR,
    collection_name="papers_rag",
)
print(f"  ✓ Chroma persistente · {len(chunks)} vectores · {PERSIST_DIR}")

  ✓ Chroma persistente · 283 vectores · ./rag_chroma_db


## 5. Retrieval base · top-k

El retriever clásico: top-k por similitud coseno. Para esta demo recuperamos top-20 (luego el reranker se queda con los 3 mejores).

In [8]:
base_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 20},
)

# Smoke test
hits = base_retriever.invoke("What is multi-head attention?")
print(f"top-20 recuperados, mostramos los 3 primeros:\n")
for i, h in enumerate(hits[:3], 1):
    print(f"#{i} · {h.metadata['source']} · pg {h.metadata.get('page', '?')}")
    print(f"     {h.page_content[:150]}...\n")

top-20 recuperados, mostramos los 3 primeros:

#1 · attention.pdf · pg 4
     output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
Multi-head attention allows th...

#2 · attention.pdf · pg 3
     Scaled Dot-Product Attention
 Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
att...

#3 · attention.pdf · pg 4
     The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder...



## 6. Retrieval con filtro de metadata

Ejemplo de cómo la metadata habilita queries más precisas. "Buscar solo en el paper Self-RAG" se hace con un filtro, sin re-indexar nada.

In [9]:
filtered_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5, "filter": {"source": "selfrag.pdf"}},
)

hits = filtered_retriever.invoke("How does self-reflection work?")
print(f"resultados solo de selfrag.pdf:\n")
for i, h in enumerate(hits, 1):
    print(f"#{i} · {h.metadata['source']} · pg {h.metadata.get('page', '?')}")
    print(f"     {h.page_content[:130]}...\n")

resultados solo de selfrag.pdf:

#1 · selfrag.pdf · pg 2
     through self-reflection, without relying on external models at inference. Moreover, our self-reflection
mechanism also evaluates o...

#2 · selfrag.pdf · pg 16
     Preprint.
A S ELF -R AG DETAILS
A.1 R EFLECTION TOKENS .
Definitions of reflection tokens. Below, we provide a detailed definition...

#3 · selfrag.pdf · pg 3
     Training overview. SELF -RAG enables an arbitrary LM to generate text with reflection tokens
by unifying them as next token predic...

#4 · selfrag.pdf · pg 2
     generation guided by reflections tokens to further improve generation quality and attributions.
Concurrent RAG work. A few concurr...

#5 · selfrag.pdf · pg 0
     retrieve passages indiscriminately regardless of whether the factual grounding is helpful. Moreover,
the output is not guaranteed ...



## 7. Reranker · Cohere `rerank-v3.5`

El retriever es **rápido pero aproximado** (bi-encoder). El reranker es **lento pero preciso** (cross-encoder, mira query y documento juntos).

Patrón productivo: recuperar amplio (top-20) y rerankear fino (top-3).

In [11]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(
    model="rerank-v3.5",
    top_n=3,
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)

query = "What is multi-head attention?"

print("=== Sin reranker (top-3 directo) ===")
for i, h in enumerate(base_retriever.invoke(query)[:3], 1):
    print(f"#{i} · {h.metadata['source']} · pg {h.metadata.get('page', '?')}")

print("\n=== Con reranker (top-20 → top-3) ===")
for i, h in enumerate(compression_retriever.invoke(query), 1):
    print(f"#{i} · {h.metadata['source']} · pg {h.metadata.get('page', '?')}")

=== Sin reranker (top-3 directo) ===
#1 · attention.pdf · pg 4
#2 · attention.pdf · pg 3
#3 · attention.pdf · pg 4

=== Con reranker (top-20 → top-3) ===
#1 · attention.pdf · pg 4
#2 · attention.pdf · pg 3
#3 · attention.pdf · pg 1


## 8. Generación con fuentes citadas · LCEL

Cadena clásica: query → retrieve+rerank → prompt con contexto → LLM → respuesta. Forzamos al modelo a citar las páginas y archivos en su respuesta.

In [12]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-5-mini")

prompt = ChatPromptTemplate.from_template(
    """Responde a la pregunta usando SOLO el contexto.
Si la respuesta no está en el contexto, di 'no aparece en los documentos'.
Cita siempre las fuentes con el formato [source · pg N].

Contexto:
{context}

Pregunta: {question}
"""
)

def format_context(docs):
    parts = []
    for d in docs:
        source = d.metadata.get("source", "?")
        page = d.metadata.get("page", "?")
        parts.append(f"[{source} · pg {page}]\n{d.page_content}")
    return "\n\n---\n\n".join(parts)

rag_chain = (
    {
        "context": compression_retriever | format_context,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

### 8.1. Demo · 3 queries

In [13]:
queries = [
    "What is the difference between RAG and Self-RAG?",
    "Explain multi-head attention in 3 sentences.",
    "Who are the authors of the original Transformer paper?",
]

for q in queries:
    print(f"\n{'='*70}\nQ: {q}\n{'='*70}")
    answer = rag_chain.invoke(q)
    print(answer)


Q: What is the difference between RAG and Self-RAG?
SELF‑RAG difiere de RAG en varios aspectos clave:

- Recuperación bajo demanda frente a recuperación fija: RAG recupera siempre un número fijo de documentos aunque no sean necesarios; SELF‑RAG primero decide si la recuperación ayudaría y sólo entonces emite un token de recuperación que llama al recuperador (recuperación on‑demand). [selfrag.pdf · pg 0]

- Revisión y selección mediante autorreflexión: RAG no vuelve a evaluar la calidad de la generación, mientras que SELF‑RAG genera "tokens de crítica" para criticar sus propias salidas y elegir la mejor en términos de factualidad y calidad. [selfrag.pdf · pg 0]

- Procesamiento y filtrado interno de pasajes: SELF‑RAG procesa varios pasajes recuperados en paralelo y filtra los irrelevantes mediante autorreflexión sin depender de modelos externos en inferencia; RAG convencional no realiza este proceso. [selfrag.pdf · pg 2]

- Citas y verificación más explícitas: SELF‑RAG proporciona cita

## 9. Inspección · qué chunks están alimentando la respuesta

Buen hábito: nunca confíes en la respuesta sin ver qué contexto recibió el modelo. Esto te ayuda a debuggear retrieval malo (la causa nº1 de RAGs que mienten).

In [14]:
def debug_query(query: str):
    docs = compression_retriever.invoke(query)
    print(f"Q: {query}\n")
    print(f"Contexto recuperado ({len(docs)} chunks):\n")
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source", "?")
        pg = d.metadata.get("page", "?")
        print(f"  #{i} · {src} · pg {pg}")
        print(f"      {d.page_content[:200]}...\n")
    print("\n--- Respuesta del LLM ---")
    print(rag_chain.invoke(query))

debug_query("What is Self-RAG?")

Q: What is Self-RAG?

Contexto recuperado (3 chunks):

  #1 · selfrag.pdf · pg 2
      guided decoding framework, but they focus only on reasoning tasks with one evaluation dimension
(reasoning path consistency) and without retrieval. Recent work on LLM refinement (Dhuliawala
et al., 20...

  #2 · selfrag.pdf · pg 1
      Prompt +  
Retrieve
Figure 1: Overview of SELF -RAG. SELF -RAG learns to retrieve, critique, and generate text passages
to enhance overall generation quality, factuality, and verifiability.
consistent...

  #3 · selfrag.pdf · pg 9
      predicts irrelevant or no support . We then ask our annotators whether the model-predicted
reflection tokens about ISREL and ISSUP match their inspections (e.g., whether the fully supported
output is ...


--- Respuesta del LLM ---
Self-RAG (Self-Reflective Retrieval-Augmented Generation) es un marco que mejora la calidad y la factualidad de un LLM mediante recuperación bajo demanda y autorreflexión, sin sacrificar la creatividad y ver

## 9b. Hybrid search · BM25 + dense + RRF

El retriever que llevamos hasta ahora es **dense-only** (similitud coseno sobre embeddings). Pero hay queries donde **keywords literales** importan más que la semántica:

- nombres propios: "Vaswani", "Lewis et al."
- siglas: "BERT", "FAISS"
- jerga muy específica

**Solución**: combinar dense + BM25 (keyword classic) y fusionar resultados con **Reciprocal Rank Fusion** (RRF). Es 5 líneas y suele dar +5-10pp de recall.

Patrón usado en producción por Anthropic, Vespa, Elasticsearch, etc.

In [17]:
%pip install -q \
  langchain-classic \
  langchain-community \
  rank_bm25

In [18]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# 1. BM25 sobre los chunks
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 20

# 2. Retriever híbrido (dense + sparse)
hybrid_retriever = EnsembleRetriever(
    retrievers=[base_retriever, bm25_retriever],
    weights=[0.6, 0.4],
)

# 3. Reranker encima
hybrid_compression = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=hybrid_retriever,
)

query = "What did Vaswani et al. propose in 2017?"

print("=== Solo dense (top-3 con reranker) ===")

for i, h in enumerate(compression_retriever.invoke(query), 1):
    print(
        f"#{i} · {h.metadata['source']} · "
        f"pg {h.metadata.get('page', '?')}"
    )

print("\n=== Hybrid (dense + BM25) + reranker ===")

for i, h in enumerate(hybrid_compression.invoke(query), 1):
    print(
        f"#{i} · {h.metadata['source']} · "
        f"pg {h.metadata.get('page', '?')}"
    )

=== Solo dense (top-3 con reranker) ===
#1 · attention.pdf · pg 0
#2 · rag.pdf · pg 14
#3 · rag.pdf · pg 14

=== Hybrid (dense + BM25) + reranker ===
#1 · attention.pdf · pg 0
#2 · rag.pdf · pg 14
#3 · rag.pdf · pg 14


**Cuándo usar hybrid**:

- Cuando tu corpus tiene jerga específica, IDs, nombres propios
- Cuando dense retrieval por sí solo da resultados "semánticos pero off-topic"
- Cuando el usuario espera coincidencia exacta de keywords (búsqueda enterprise)

**Cuándo no**: si tu corpus es prosa fluida y el usuario hace preguntas naturales, dense suele bastar y BM25 añade ruido.

## 10. ¿Y la calidad?

Hasta aquí tenemos un RAG **funcional**. La pregunta es ¿cómo de bueno es?

Métricas estándar para RAG:

| Métrica | Qué mide |
|---|---|
| **Faithfulness** | ¿La respuesta se ciñe al contexto recuperado? (no alucina) |
| **Answer relevancy** | ¿Responde realmente a la pregunta? |
| **Context precision** | ¿Los chunks recuperados son relevantes? |
| **Context recall** | ¿El retriever cubre toda la info necesaria? |

Estas se calculan con **RAGas** o **LLM-as-judge**. Lo veremos a fondo en **M20** (eval con RAGas) + **M21** (Langfuse), donde instrumentamos este mismo notebook para observabilidad continua.

## Cierre

- Has construido un RAG **completo**: loader → chunker → embedder → store → retriever + reranker → LLM
- Has visto que la **metadata** habilita filtros y citaciones
- Has visto que el **reranker** mejora precisión sin reentrenar nada
- Has visto que el patrón LCEL hace el pipeline **legible y componible**
- En M20 entras a **medir** este pipeline con rigor (RAGas) · y en M21 lo instrumentas con Langfuse